# Arrow filesystems - Rust

All 8 Rust examples from [docs/arrowfs.md](https://platob.github.io/yggdryl/arrowfs/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

## Construct from a filesystem and a path

In [ ]:
use std::sync::Arc;

use yggdryl::arrowfs::{File, MemoryFileSystem};
use yggdryl::io::IOBase;

let filesystem = Arc::new(MemoryFileSystem::new());
let mut handle = File::from_location(filesystem, "bucket/trades.bin")?;

// Per the laziness contract nothing exists until something is written.
assert!(!handle.exists());
assert_eq!(handle.read_all_bytes()?, b"");

handle.write_all_bytes(b"AAPL")?;
handle.close()?;
assert_eq!(handle.read_all_bytes()?, b"AAPL");

// The handle's identity is a canonical URL naming the filesystem.
assert_eq!(handle.url().to_string(), "memory://bucket/trades.bin");

## A positional write publishes when the handle closes

In [ ]:
use std::sync::Arc;

use yggdryl::arrowfs::{ArrowFileSystem, File, MemoryFileSystem};
use yggdryl::io::IOBase;

let filesystem = Arc::new(MemoryFileSystem::new());
filesystem.write_full("bucket/trades.bin", b"stored")?;
let mut handle = File::from_location(filesystem.clone(), "bucket/trades.bin")?;

// Positional writes are pieces of a value, so they stage.
handle.truncate(0)?;
handle.pwrite(0, b"pend")?;
handle.pwrite(4, b"ing")?;

// The handle presents the pending value; the filesystem still has the old one.
assert_eq!(handle.read_all_bytes()?, b"pending");
assert_eq!(filesystem.file_info("bucket/trades.bin")?.size, 6);

handle.close()?;
assert_eq!(filesystem.file_info("bucket/trades.bin")?.size, 7);

## Folders, globs, and partitions

In [ ]:
use std::sync::Arc;

use yggdryl::arrowfs::{ArrowFileSystem, Folder, MemoryFileSystem};
use yggdryl::io::IOBase;

let filesystem = Arc::new(MemoryFileSystem::new());
for year in ["2024", "2025"] {
    let leaf = format!("bucket/year={year}/part-0.parquet");
    filesystem.write_full(&leaf, b"PAR1")?;
}
let lake = Folder::from_location(filesystem, "bucket")?;

assert!(lake.is_container());
assert_eq!(lake.ls(false, false).count(), 2);
assert_eq!(lake.glob("**/*.parquet", false)?.count(), 2);

// A fixed prefix is descended rather than listed and filtered.
assert_eq!(lake.glob("year=2024/**/*.parquet", false)?.count(), 1);

// Hive pairs are read off the location, as they are for any backend.
let selected: Vec<_> = lake
    .children_where(&[("year", "2024")], false)?
    .collect::<yggdryl::Result<_>>()?;
assert_eq!(selected.len(), 1);

## Records

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::arrowfs::{File, MemoryFileSystem};
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{IOBase, IOMedia};
use yggdryl::DataType;

let schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("row");

let batch = RecordBatch::try_new(
    schema.clone().into_arrow_schema()?,
    vec![
        Arc::new(Int64Array::from(vec![1, 2])),
        Arc::new(StringArray::from(vec![Some("AAPL"), None])),
    ],
)?;

let filesystem = Arc::new(MemoryFileSystem::new());
let mut handle = File::from_location(filesystem, "bucket/trades.parquet")?;
let options = handle.record_options()?.with_field(schema.clone());

handle.overwrite_arrow_reader(
    yggdryl::arrow::batch_reader(batch.schema(), [batch]),
    &options,
)?;
handle.close()?;

let rows: usize = handle
    .read_arrow_reader(&options)?
    .map(|batch| batch.unwrap().num_rows())
    .sum();
assert_eq!(rows, 2);
assert_eq!(handle.read_arrow_field(&options)?, schema);

## Composing with the wrappers

In [ ]:
use std::sync::Arc;

use yggdryl::arrowfs::{File, MemoryFileSystem};
use yggdryl::io::{Coded, IOBase};
use yggdryl::{Codec, MimeType};

let filesystem = Arc::new(MemoryFileSystem::new());
let leaf = File::from_location(filesystem.clone(), "bucket/trades.json.gz")?;

let mut coded = Coded::new(leaf, Codec::Gzip);
coded.write_all_bytes(br#"{"symbol":"AAPL"}"#)?;
coded.close()?;

// The view presents the decoded value...
assert_eq!(coded.media_type().base(), &MimeType::JSON);
assert_eq!(coded.read_all_bytes()?, br#"{"symbol":"AAPL"}"#);

// ...while what the filesystem holds is gzip.
let stored = File::from_location(filesystem, "bucket/trades.json.gz")?;
assert_eq!(&stored.read_all_bytes()?[..2], &[0x1f, 0x8b]);

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::arrowfs::{Folder, MemoryFileSystem};
use yggdryl::iceberg::{FormatVersion, PartitionSpec, Table};
use yggdryl::io::{IOBase, IOMedia};
use yggdryl::DataType;

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");
let batch = RecordBatch::try_new(
    schema.clone().into_arrow_schema()?,
    vec![Arc::new(Int64Array::from(vec![1, 2]))],
)?;

let filesystem = Arc::new(MemoryFileSystem::new());
let root = Folder::from_location(filesystem, "warehouse/trades")?;

let mut table = Table::create(
    root,
    FormatVersion::V2,
    schema,
    PartitionSpec::unpartitioned(),
)?;
table.append(yggdryl::arrow::batch_reader(batch.schema(), [batch]))?;

let options = table.record_options()?;
let rows: usize = table
    .read_arrow_reader(&options)?
    .map(|batch| batch.unwrap().num_rows())
    .sum();
assert_eq!(rows, 2);

## The two filesystems that ship here

In [ ]:
use std::sync::Arc;

use yggdryl::arrowfs::{File, LocalFileSystem};
use yggdryl::io::IOBase;

let root = std::env::temp_dir().join(format!("yggdryl-doc-arrowfs-{}", std::process::id()));
std::fs::create_dir_all(&root)?;
let location = root.join("trades.bin").to_string_lossy().replace('\\', "/");

let mut handle = File::from_location(Arc::new(LocalFileSystem::new()), &location)?;
handle.write_all_bytes(b"AAPL")?;
handle.close()?;

assert_eq!(std::fs::read(root.join("trades.bin"))?, b"AAPL");
let _ = std::fs::remove_dir_all(&root);

## Bringing your own filesystem

In [ ]:
use std::sync::Arc;

use yggdryl::arrowfs::{ArrowFileSystem, FileInfo, File};
use yggdryl::io::IOBase;
use yggdryl::Result;

/// A filesystem holding exactly one read-only object.
struct OneObject;

impl ArrowFileSystem for OneObject {
    fn type_name(&self) -> &str {
        "memory"
    }

    fn file_info(&self, path: &str) -> Result<FileInfo> {
        Ok(if path == "bucket/only.bin" {
            FileInfo::file(path, 5)
        } else {
            FileInfo::not_found(path)
        })
    }

    fn list(&self, _path: &str, _recursive: bool) -> yggdryl::arrowfs::FileInfos {
        yggdryl::arrowfs::FileInfos::new(
            [Ok(FileInfo::file("bucket/only.bin", 5))].into_iter(),
        )
    }

    fn read_range(&self, path: &str, offset: u64, buffer: &mut [u8]) -> Result<usize> {
        if path != "bucket/only.bin" {
            return Ok(0);
        }
        let value = b"AAPL!";
        let offset = offset as usize;
        if offset >= value.len() {
            return Ok(0);
        }
        let count = (value.len() - offset).min(buffer.len());
        buffer[..count].copy_from_slice(&value[offset..offset + count]);
        Ok(count)
    }

    fn write_full(&self, _path: &str, _bytes: &[u8]) -> Result<()> {
        Ok(())
    }

    fn create_dir(&self, _path: &str) -> Result<()> {
        Ok(())
    }

    fn delete_file(&self, _path: &str) -> Result<()> {
        Ok(())
    }
}

let handle = File::from_location(Arc::new(OneObject), "bucket/only.bin")?;
assert_eq!(handle.read_all_bytes()?, b"AAPL!");
assert_eq!(handle.read_range(1, 3)?, b"APL");